In [ ]:
import torch
import pandas as pd
from collections import defaultdict
from time import time
from tqdm import tqdm

class HVPTester:
    def __init__(self, test_grad_dict, tr_grad_dict, n_train):
        self.test_grad_dict = test_grad_dict
        self.tr_grad_dict = tr_grad_dict
        self.n_train = n_train
        self.hvp_dict = {}
        self.time_dict = {}
        self.IF_dict = {}
        
    def compute_hvp_proposed_slow(self, lambda_const_param=10):
        start_time = time()
        hvp_proposed_dict=defaultdict(dict)
        for val_id in tqdm(self.test_grad_dict.keys(), desc="Slow HVP"):
            for weight_name in self.test_grad_dict[val_id]:
                S=torch.zeros(len(self.tr_grad_dict.keys()), dtype=torch.float64)
                for tr_id in self.tr_grad_dict:
                    tmp_grad = self.tr_grad_dict[tr_id][weight_name]
                    S[tr_id]=torch.mean(tmp_grad**2)
                lambda_const = torch.mean(S) / lambda_const_param

                hvp=torch.zeros(self.test_grad_dict[val_id][weight_name].shape, dtype=torch.float64)
                for tr_id in self.tr_grad_dict:
                    tmp_grad = self.tr_grad_dict[tr_id][weight_name]
                    C_tmp = torch.sum(self.test_grad_dict[val_id][weight_name] * tmp_grad) / (lambda_const + torch.sum(tmp_grad**2))
                    hvp += (self.test_grad_dict[val_id][weight_name] - C_tmp*tmp_grad) / (self.n_train*lambda_const)
                hvp_proposed_dict[val_id][weight_name] = hvp
        self.hvp_dict['proposed_slow'] = hvp_proposed_dict
        self.time_dict['proposed_slow'] = time()-start_time
    
    def compute_hvp_proposed(self, lambda_const_param=10):
        start_time = time()
        hvp_proposed_dict = defaultdict(dict)
        
        tr_ids = list(self.tr_grad_dict.keys())
        n_train = self.n_train
        
        tr_grads_stacked = {}
        for weight_name in next(iter(self.tr_grad_dict.values())).keys():
            grads = [self.tr_grad_dict[tr_id][weight_name].flatten() for tr_id in tr_ids]
            tr_grads_stacked[weight_name] = torch.stack(grads).to(torch.float64)
        
        for val_id in tqdm(self.test_grad_dict.keys(), desc="Fast HVP"):
            for weight_name in self.test_grad_dict[val_id]:
                val_grad = self.test_grad_dict[val_id][weight_name].flatten().to(torch.float64)
             
                S = torch.mean(tr_grads_stacked[weight_name] ** 2, dim=1)
                lambda_const = torch.mean(S) / lambda_const_param
                
                dot_products = torch.matmul(tr_grads_stacked[weight_name], val_grad)
                denom = lambda_const + torch.sum(tr_grads_stacked[weight_name] ** 2, dim=1)
                C_tmp = dot_products / denom
                
                weighted_terms = (val_grad.unsqueeze(0) - (C_tmp.unsqueeze(1) * tr_grads_stacked[weight_name]))
                hvp = weighted_terms.sum(dim=0) / (n_train * lambda_const)
                
                hvp = hvp.view(self.test_grad_dict[val_id][weight_name].shape)
                hvp_proposed_dict[val_id][weight_name] = hvp
        
        self.hvp_dict['proposed'] = hvp_proposed_dict
        self.time_dict['proposed'] = time() - start_time
    
    def compute_IF_slow(self):
        self.IF_dict = {}
        for method_name in self.hvp_dict:
            print(f"Computing IF (slow) for method: {method_name}")
            if_tmp_dict = defaultdict(dict)
            for tr_id in self.tr_grad_dict:
                for val_id in self.test_grad_dict:
                    if_tmp_value = torch.tensor(0.0, dtype=torch.float64)
                    for weight_name in self.test_grad_dict[0]:
                        hvp_vec = self.hvp_dict[method_name][val_id][weight_name]
                        tr_grad_vec = self.tr_grad_dict[tr_id][weight_name]
                        if_tmp_value += torch.sum(hvp_vec * tr_grad_vec)
                    if_tmp_dict[tr_id][val_id] = -if_tmp_value.item()
            self.IF_dict[method_name + '_slow'] = pd.DataFrame(if_tmp_dict, dtype=float)
    
    def compute_IF(self):
        self.IF_dict = {}
        for method_name in self.hvp_dict:
            print(f"Computing IF (fast) for method: {method_name}")
            tr_ids = list(self.tr_grad_dict.keys())
            val_ids = list(self.test_grad_dict.keys())
            
            tr_grads_stacked = {}
            for weight_name in next(iter(self.tr_grad_dict.values())).keys():
                grads = [self.tr_grad_dict[tr_id][weight_name].flatten().to(torch.float64) for tr_id in tr_ids]
                tr_grads_stacked[weight_name] = torch.stack(grads)
            
            hvp_stacked = {}
            for weight_name in next(iter(self.test_grad_dict.values())).keys():
                grads = [self.hvp_dict[method_name][val_id][weight_name].flatten().to(torch.float64) for val_id in val_ids]
                hvp_stacked[weight_name] = torch.stack(grads)
            
            IF_matrix = torch.zeros(len(tr_ids), len(val_ids), dtype=torch.float64)
            for weight_name in hvp_stacked:
                dot_products = torch.matmul(tr_grads_stacked[weight_name], hvp_stacked[weight_name].T)
                IF_matrix += dot_products
            
            IF_matrix = -IF_matrix
            self.IF_dict[method_name] = pd.DataFrame(IF_matrix.numpy(), index=tr_ids, columns=val_ids)

def generate_mock_data(n_train=5000, n_val=300, grad_dim=768):
    tr_grad_dict = {}
    for tr_id in range(n_train):
        tr_grad_dict[tr_id] = {'weight': torch.randn(grad_dim, dtype=torch.float64)}
    test_grad_dict = {}
    for val_id in range(n_val):
        test_grad_dict[val_id] = {'weight': torch.randn(grad_dim, dtype=torch.float64)}
    return test_grad_dict, tr_grad_dict

# --- Run everything ---

test_grad_dict, tr_grad_dict = generate_mock_data()

tester = HVPTester(test_grad_dict, tr_grad_dict, n_train=len(tr_grad_dict))

tester.compute_hvp_proposed_slow()
tester.compute_hvp_proposed()

tester.compute_IF_slow()
tester.compute_IF()

# Compare IF matrices
for method in tester.IF_dict:
    print(f"\nMethod: {method}")
    df = tester.IF_dict[method]
    print(df)

# Compare slow vs fast IF for 'proposed' method
df_slow = tester.IF_dict['proposed_slow']
df_fast = tester.IF_dict['proposed']

max_diff = (df_slow.values - df_fast.values).max()
print(f"\nMax absolute difference between slow and fast IF matrices: {max_diff}")

if max_diff < 1e-12:
    print("Success! IF matrices MATCH closely with high precision.")
else:
    print("Warning! IF matrices differ.")

# Also compare slow vs fast HVP for reference
for val_id in test_grad_dict:
    for w_name in test_grad_dict[val_id]:
        hvp_slow = tester.hvp_dict['proposed_slow'][val_id][w_name]
        hvp_fast = tester.hvp_dict['proposed'][val_id][w_name]
        assert torch.allclose(hvp_slow, hvp_fast, atol=1e-10), "HVP outputs differ!"

print("\nAll tests completed with float64 precision.")


Slow HVP:  13%|█▎        | 39/300 [00:04<00:30,  8.50it/s]

Slow HVP:  13%|█▎        | 39/300 [00:04<00:30,  8.67it/s]


KeyboardInterrupt: 

In [ ]:
# test_hvp_equivalence.ipynb

import torch
from collections import defaultdict
from time import time
from tqdm import tqdm

# Mock class to hold the methods and variables
class HVPTester:
    def __init__(self, test_grad_dict, tr_grad_dict, n_train):
        self.test_grad_dict = test_grad_dict
        self.tr_grad_dict = tr_grad_dict
        self.n_train = n_train
        self.hvp_dict = {}
        self.time_dict = {}
        
    def compute_hvp_proposed_slow(self, lambda_const_param=10):
        start_time = time()
        hvp_proposed_dict=defaultdict(dict)
        for val_id in tqdm(self.test_grad_dict.keys()):
            for weight_name in self.test_grad_dict[val_id]:
                # lambda_const computation
                S=torch.zeros(len(self.tr_grad_dict.keys()))
                for tr_id in self.tr_grad_dict:
                    tmp_grad = self.tr_grad_dict[tr_id][weight_name]
                    S[tr_id]=torch.mean(tmp_grad**2)
                lambda_const = torch.mean(S) / lambda_const_param # layer-wise lambda

                # hvp computation
                hvp=torch.zeros(self.test_grad_dict[val_id][weight_name].shape)
                for tr_id in self.tr_grad_dict:
                    tmp_grad = self.tr_grad_dict[tr_id][weight_name]
                    C_tmp = torch.sum(self.test_grad_dict[val_id][weight_name] * tmp_grad) / (lambda_const + torch.sum(tmp_grad**2))
                    hvp += (self.test_grad_dict[val_id][weight_name] - C_tmp*tmp_grad) / (self.n_train*lambda_const)
                hvp_proposed_dict[val_id][weight_name] = hvp
        self.hvp_dict['proposed_slow'] = hvp_proposed_dict
        self.time_dict['proposed_slow'] = time()-start_time
    def compute_hvp_proposed(self, lambda_const_param=10):
        start_time = time()
        hvp_proposed_dict = defaultdict(dict)
        
        tr_ids = list(self.tr_grad_dict.keys())
        n_train = self.n_train
        

        tr_grads_stacked = {}
        for weight_name in next(iter(self.tr_grad_dict.values())).keys():
            grads = [self.tr_grad_dict[tr_id][weight_name].flatten() for tr_id in tr_ids]
            tr_grads_stacked[weight_name] = torch.stack(grads)  # shape (n_train, grad_dim)
        
        for val_id in tqdm(self.test_grad_dict.keys()):
            for weight_name in self.test_grad_dict[val_id]:
                val_grad = self.test_grad_dict[val_id][weight_name].flatten()  # shape (grad_dim,)
             
                S = torch.mean(tr_grads_stacked[weight_name] ** 2, dim=1)  # (n_train,)
                lambda_const = torch.mean(S) / lambda_const_param  # scalar
                
             
                dot_products = torch.matmul(tr_grads_stacked[weight_name], val_grad)  # (n_train,)
                denom = lambda_const + torch.sum(tr_grads_stacked[weight_name] ** 2, dim=1)  # (n_train,)
                C_tmp = dot_products / denom  # (n_train,)
                
              
                weighted_terms = (val_grad.unsqueeze(0) - (C_tmp.unsqueeze(1) * tr_grads_stacked[weight_name]))  # (n_train, grad_dim)
                hvp = weighted_terms.sum(dim=0) / (n_train * lambda_const)  # (grad_dim,)
                
           
                hvp = hvp.view(self.test_grad_dict[val_id][weight_name].shape)
                hvp_proposed_dict[val_id][weight_name] = hvp
        
        self.hvp_dict['proposed'] = hvp_proposed_dict
        self.time_dict['proposed'] = time() - start_time


# Mock data generation function
def generate_mock_data(n_train=5, n_val=3, weight_shapes={"w1": (2,2), "w2": (3,)}):
    tr_grad_dict = {}
    for tr_id in range(n_train):
        tr_grad_dict[tr_id] = {}
        for w_name, shape in weight_shapes.items():
            tr_grad_dict[tr_id][w_name] = torch.randn(768)
            
    test_grad_dict = {}
    for val_id in range(n_val):
        test_grad_dict[val_id] = {}
        for w_name, shape in weight_shapes.items():
            test_grad_dict[val_id][w_name] = torch.randn(768)
    
    return test_grad_dict, tr_grad_dict

# Generate mock data
test_grad_dict, tr_grad_dict = generate_mock_data(n_train=100, n_val=100)

# Create tester instance
tester = HVPTester(test_grad_dict, tr_grad_dict, n_train=len(tr_grad_dict))

# Run slow method
tester.compute_hvp_proposed_slow()

# Run fast method
tester.compute_hvp_proposed()

# Compare results
for val_id in test_grad_dict:
    for w_name in test_grad_dict[val_id]:
        hvp_slow = tester.hvp_dict['proposed_slow'][val_id][w_name]
        hvp_fast = tester.hvp_dict['proposed'][val_id][w_name]
        if torch.allclose(hvp_slow, hvp_fast):
            print(f"Val {val_id}, Weight {w_name}: Outputs MATCH.")
        else:
            print(f"Val {val_id}, Weight {w_name}: Outputs DIFFER.")
            print("Slow output:", hvp_slow)
            print("Fast output:", hvp_fast)

print(f"Slow method time: {tester.time_dict['proposed_slow']:.4f} seconds")
print(f"Fast method time: {tester.time_dict['proposed']:.4f} seconds")
test_grad_dict, tr_grad_dict = generate_mock_data(n_train=100000, n_val=1000)

# Create tester instance
tester = HVPTester(test_grad_dict, tr_grad_dict, n_train=len(tr_grad_dict))


tester = HVPTester(test_grad_dict, tr_grad_dict, n_train=len(tr_grad_dict))


# Run fast method
tester.compute_hvp_proposed()
